In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import PositionalSharding
from functools import partial
import time

# --- Configuration (Extreme Scaling)
MAX_RECURSION_DEPTH = 50_000  # 🔥 Going up to 50,000!
OPTIMAL_DEPTH_STEP = 10_000  # 🔥 Executing in 10,000-depth chunks
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 10_000_000  # 🔥 10 Million Samples!

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """🔥 Executes in 10,000-depth chunks to maximize TPU efficiency"""
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# --- TPU Sharding Setup
devices = jax.devices()
sharding = PositionalSharding(devices)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ Extreme Scaling with Depth 50,000
def process_with_extreme_depths(x, total_depth):
    """🔥 Instead of smaller steps, we now execute in 10,000-depth chunks"""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# --- Execute at Maximum Performance
for depth in [10_000, 25_000, 50_000]:  # 🔥 Pushing TPU with insane depths
    output_batch = process_with_extreme_depths(batch_input, depth)
    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

NUM_TRIALS = 3  # 🔥 Reduce trials to avoid unnecessary overhead

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

for depth in [10_000, 25_000, 50_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_extreme_depths(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# --- Investigate TPU Compilation Stability
compiled_fn_10k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=10_000)
compiled_fn_50k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=50_000)

print("\n🚀 XLA Compilation for Depth=10,000:")
print(compiled_fn_10k.as_text())

print("\n🚀 XLA Compilation for Depth=50,000:")
print(compiled_fn_50k.as_text())





Batch Output Shape (Depth=10000): (10000000,)
Batch Output Shape (Depth=25000): (10000000,)
Batch Output Shape (Depth=50000): (10000000,)

🔥 TPU Benchmark (Depth=10000, Batch=10000000)
Avg: 1.028347, Min: 0.341236, Max: 2.401967

🔥 TPU Benchmark (Depth=25000, Batch=10000000)
Avg: 0.608584, Min: 0.606948, Max: 0.609815

🔥 TPU Benchmark (Depth=50000, Batch=10000000)
Avg: 1.408481, Min: 1.402873, Max: 1.417716

🚀 XLA Compilation for Depth=10,000:
module @jit_dppu_with_dynamic_pi_phi attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<10000000xf32> {mhlo.layout_mode = "default"}, %arg1: tensor<i32> {mhlo.layout_mode = "default"}) -> (tensor<10000000xf32> {jax.result_info = "", mhlo.layout_mode = "default"}) {
    %0 = call @dppu_with_dynamic_pi_phi(%arg1, %arg0) : (tensor<i32>, tensor<10000000xf32>) -> tensor<10000000xf32>
    return %0 : tensor<10000000xf32>
  }
  func.func private @dppu_with_dynamic_pi_phi(%arg0: tensor<i32> {m

/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
